In [ ]:
# cell 1
# Install runtime dependencies for GPT-OSS + vLLM on Colab / NVIDIA A100 80GB.

!pip -q install -U uv

!uv pip install --system -U openai requests tqdm jsonschema psutil numpy pandas accelerate safetensors huggingface_hub

# Remove optional packages that may break imports or pull mismatched CUDA wheels.
!uv pip uninstall --system -y torchcodec torchvision torchaudio sentence-transformers || true

# Transformers is used for tokenizer-based input truncation.
!uv pip install --system -U "transformers>=4.56.0"

# A100 is Ampere, not Blackwell.
# Do not force CUDA 13 / cu130 Blackwell wheels on A100.
# vLLM recommends the auto backend for current releases.
!uv pip install --system -U vllm --torch-backend=auto

# Optional fallback only if the official vLLM install above fails in your Colab runtime.
# !uv pip install --system --pre -U "vllm==0.10.1+gptoss" \
#     --extra-index-url https://wheels.vllm.ai/gpt-oss/ \
#     --extra-index-url https://download.pytorch.org/whl/nightly/cu128 \
#     --index-strategy unsafe-best-match

import sys
import importlib.metadata as md

import torch
import vllm
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)
print("transformers:", transformers.__version__)

for pkg in ["torchcodec", "torchvision", "torchaudio", "sentence-transformers"]:
    try:
        print(pkg + ":", md.version(pkg))
    except Exception:
        print(pkg + ": not installed")

!nvidia-smi

Using Python 3.12.13 environment at: /usr
Resolved 71 packages in 167ms
Prepared 8 packages in 0.92ms
Uninstalled 8 packages in 256ms
Installed 8 packages in 214ms
 - numpy==2.3.5
 + numpy==2.4.6
 - nvidia-cublas==13.1.0.3
 + nvidia-cublas==13.1.1.3
 - nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cudnn-cu13==9.20.0.48
 - nvidia-cusparselt-cu13==0.8.0
 + nvidia-cusparselt-cu13==0.8.1
 - nvidia-nccl-cu13==2.28.9
 + nvidia-nccl-cu13==2.29.7
 - setuptools==80.10.2
 + setuptools==81.0.0
 - torch==2.11.0+cu130
 + torch==2.12.0
 - triton==3.6.0
 + triton==3.7.0
Using Python 3.12.13 environment at: /usr
Uninstalled 2 packages in 110ms
 - torchaudio==2.11.0+cu130
 - torchvision==0.26.0+cu130
Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 102ms
Checked 27 packages in 0.62ms
Using Python 3.12.13 environment at: /usr
Resolved 190 packages in 6.48s
Prepared 10 packages in 12ms
Uninstalled 8 packages in 221ms
Installed 10 packages in 212ms
 - numpy==2.4.6
 + numpy==2.3.5
 - nvidia-cubla

In [ ]:
# cell 2
# Imports and global configuration for BM25 evidence + GPT-OSS on A100 80GB.

import os
import re
import gc
import json
import time
import shlex
import shutil
import psutil
import subprocess
import traceback
import site
import glob

from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from transformers import AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# GPT-OSS model.
LLM_MODEL_NAME = "openai/gpt-oss-120b"

# GPT-OSS supports low / medium / high.
# Use "high" if you want the strongest reasoning behavior.
# For faster runs on A100, you can change this to "medium".
REASONING_EFFORT = "high"

PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# The prompt sent to the model is capped around 8192 tokens.
MAX_INPUT_TOKENS = 8192

# The final answer is short, but GPT-OSS reasoning tokens are included in this budget.
# 8192 is safer and faster on one A100 than 32768.
ANSWER_MAX_TOKENS = 8192

# A100 80GB can run gpt-oss-120b, but full 128k context may waste memory here.
# This is enough for MAX_INPUT_TOKENS + ANSWER_MAX_TOKENS with extra room.
MAX_MODEL_LEN = 49152

# Single A100 80GB setup.
TENSOR_PARALLEL_SIZE = 1

# Conservative memory setting for a single A100 80GB.
GPU_MEMORY_UTILIZATION = 0.95

# This notebook sends one request at a time.
MAX_NUM_SEQS = 1

# Safer value for single-GPU A100 to avoid startup or KV-cache OOM.
MAX_NUM_BATCHED_TOKENS = 1024

# Do not force fp8 KV cache on A100.
# Leave this as None so vLLM chooses the safe default.
KV_CACHE_DTYPE = None

# Disable prefix caching for reproducible evaluation behavior.
ENABLE_PREFIX_CACHING = False

# Smaller CUDA graph capture is safer on a single A100.
MAX_CUDAGRAPH_CAPTURE_SIZE = 1024

# A100 / Ampere should use TRITON_ATTN.
ATTENTION_BACKEND = "TRITON_ATTN"

# FlashInfer Blackwell-specific sampler workaround is not needed for A100.
USE_FLASHINFER_SAMPLER = None
DISABLE_FLASHINFER_COMPLETELY = False

# Add pip-installed NVIDIA CUDA libraries to LD_LIBRARY_PATH for subprocesses.
ADD_NVIDIA_PIP_LIBS_TO_LD_LIBRARY_PATH = True

SERVER_LOG_PATH = Path("/content/vllm_gpt_oss_bm25_answer_server.log")
SERVER_PID_PATH = Path("/content/vllm_gpt_oss_bm25_answer_server.pid")

PROJECT_DIR = Path("/content/drive/MyDrive/final_project")
BM25_DIR = PROJECT_DIR / "BM25"
DRIVE_EVIDENCE_DIR = BM25_DIR / "evidence"
DRIVE_ANSWER_DIR = BM25_DIR / "answer"

LOCAL_RUNTIME_DIR = Path("/content/final_project_bm25_gpt_oss_answer_copy")
LOCAL_EVIDENCE_DIR = LOCAL_RUNTIME_DIR / "evidence"
LOCAL_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "hotpotqa": {
        "drive_evidence_path": DRIVE_EVIDENCE_DIR / "hotpotqa_evidence.json",
        "local_evidence_path": LOCAL_EVIDENCE_DIR / "hotpotqa_evidence.json",
        "answer_path": DRIVE_ANSWER_DIR / "hotpotqa_bm25_gpt_oss_120b_answers.json",
    },
    "2wikimultihopqa": {
        "drive_evidence_path": DRIVE_EVIDENCE_DIR / "2wikimultihopqa_evidence.json",
        "local_evidence_path": LOCAL_EVIDENCE_DIR / "2wikimultihopqa_evidence.json",
        "answer_path": DRIVE_ANSWER_DIR / "2wikimultihopqa_bm25_gpt_oss_120b_answers.json",
    },
}

EXPECTED_NUM_RECORDS_PER_DATASET = 1000

# Use None to process all records.
ANSWER_START_INDEX = 0
ANSWER_END_INDEX = None

SAVE_EVERY_N = 1
CLEAR_CACHE_EVERY_N = 25

# Resume from existing BM25 GPT-OSS answer files if they already exist.
RESUME_IF_EXISTS = True

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print("Detected GPU:", gpu_name)
    print("Detected VRAM GB:", round(total_vram_gb, 2))

    if "A100" not in gpu_name:
        print("Warning: This configuration is tuned for NVIDIA A100 80GB.")
else:
    print("Warning: CUDA is not available.")

print("Model:", LLM_MODEL_NAME)
print("Reasoning effort:", REASONING_EFFORT)
print("Max input tokens:", MAX_INPUT_TOKENS)
print("Answer max tokens:", ANSWER_MAX_TOKENS)
print("Max model len:", MAX_MODEL_LEN)
print("Tensor parallel size:", TENSOR_PARALLEL_SIZE)
print("GPU memory utilization:", GPU_MEMORY_UTILIZATION)
print("Max num seqs:", MAX_NUM_SEQS)
print("Max num batched tokens:", MAX_NUM_BATCHED_TOKENS)
print("KV cache dtype:", KV_CACHE_DTYPE)
print("Prefix caching enabled:", ENABLE_PREFIX_CACHING)
print("Attention backend override:", ATTENTION_BACKEND)
print("BM25 directory:", BM25_DIR)
print("Evidence directory:", DRIVE_EVIDENCE_DIR)
print("Answer directory:", DRIVE_ANSWER_DIR)

for dataset_name, cfg in DATASETS.items():
    print("=" * 80)
    print("Dataset:", dataset_name)
    print("Drive evidence:", cfg["drive_evidence_path"])
    print("Local evidence:", cfg["local_evidence_path"])
    print("Answer output:", cfg["answer_path"])

Detected GPU: NVIDIA A100-SXM4-80GB
Detected VRAM GB: 79.25
Model: openai/gpt-oss-120b
Reasoning effort: high
Max input tokens: 8192
Answer max tokens: 8192
Max model len: 49152
Tensor parallel size: 1
GPU memory utilization: 0.95
Max num seqs: 1
Max num batched tokens: 1024
KV cache dtype: None
Prefix caching enabled: False
Attention backend override: TRITON_ATTN
BM25 directory: /content/drive/MyDrive/final_project/BM25
Evidence directory: /content/drive/MyDrive/final_project/BM25/evidence
Answer directory: /content/drive/MyDrive/final_project/BM25/answer
Dataset: hotpotqa
Drive evidence: /content/drive/MyDrive/final_project/BM25/evidence/hotpotqa_evidence.json
Local evidence: /content/final_project_bm25_gpt_oss_answer_copy/evidence/hotpotqa_evidence.json
Answer output: /content/drive/MyDrive/final_project/BM25/answer/hotpotqa_bm25_gpt_oss_120b_answers.json
Dataset: 2wikimultihopqa
Drive evidence: /content/drive/MyDrive/final_project/BM25/evidence/2wikimultihopqa_evidence.json
Local e

In [ ]:
# cell 3
# Mount Google Drive and copy BM25 evidence files to local Colab disk.

from google.colab import drive

MOUNTPOINT = Path("/content/drive")
drive.mount(str(MOUNTPOINT), force_remount=True)

assert PROJECT_DIR.exists(), f"PROJECT_DIR does not exist: {PROJECT_DIR}"
assert BM25_DIR.exists(), f"BM25_DIR does not exist: {BM25_DIR}"
assert DRIVE_EVIDENCE_DIR.exists(), f"Evidence directory does not exist: {DRIVE_EVIDENCE_DIR}"

DRIVE_ANSWER_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

def file_is_same_size(src: Path, dst: Path) -> bool:
    """Check whether the destination file exists and has the same size as the source."""
    return dst.exists() and dst.stat().st_size == src.stat().st_size

def copy_file_to_local(src: Path, dst: Path) -> None:
    """Copy a file to local disk using a temporary file to avoid partial copies."""
    dst.parent.mkdir(parents=True, exist_ok=True)

    if file_is_same_size(src, dst):
        print(f"Local copy already exists: {dst}")
        return

    tmp = dst.with_name(dst.name + ".tmp")

    if tmp.exists():
        tmp.unlink()

    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

    print(f"Copied to local disk: {src} -> {dst}")

for dataset_name, cfg in DATASETS.items():
    drive_path = cfg["drive_evidence_path"]
    local_path = cfg["local_evidence_path"]

    if not drive_path.exists():
        raise FileNotFoundError(f"{dataset_name}: BM25 evidence file not found: {drive_path}")

    copy_file_to_local(drive_path, local_path)

    print(f"{dataset_name}: local evidence size MB:", local_path.stat().st_size / (1024 ** 2))

print("BM25 answer directory is ready:", DRIVE_ANSWER_DIR)

Mounted at /content/drive
Copied to local disk: /content/drive/MyDrive/final_project/BM25/evidence/hotpotqa_evidence.json -> /content/final_project_bm25_gpt_oss_answer_copy/evidence/hotpotqa_evidence.json
hotpotqa: local evidence size MB: 7.00105094909668
Copied to local disk: /content/drive/MyDrive/final_project/BM25/evidence/2wikimultihopqa_evidence.json -> /content/final_project_bm25_gpt_oss_answer_copy/evidence/2wikimultihopqa_evidence.json
2wikimultihopqa: local evidence size MB: 5.920868873596191
BM25 answer directory is ready: /content/drive/MyDrive/final_project/BM25/answer


In [ ]:
#cell 4
# Load and validate evidence files.

def load_json_list(json_path: Path, dataset_name: str) -> List[Dict[str, Any]]:
    """Load a JSON file whose root must be a list."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"{dataset_name}: JSON root must be a list.")

    return data

def validate_evidence_record(record: Dict[str, Any], dataset_name: str, index: int) -> None:
    """Validate one evidence record."""
    required_keys = {"type", "question", "answer", "evidence_chunk"}

    if not isinstance(record, dict):
        raise ValueError(f"{dataset_name}: record {index} is not a dictionary.")

    missing = required_keys - set(record.keys())
    if missing:
        raise ValueError(f"{dataset_name}: record {index} is missing keys: {missing}")

    if not isinstance(record["question"], str) or not record["question"].strip():
        raise ValueError(f"{dataset_name}: record {index} has an empty question.")

    if not isinstance(record["evidence_chunk"], list):
        raise ValueError(f"{dataset_name}: record {index} evidence_chunk must be a list.")

    for j, chunk in enumerate(record["evidence_chunk"]):
        if not isinstance(chunk, dict):
            raise ValueError(f"{dataset_name}: record {index}, chunk {j} is not a dictionary.")

        if "title" not in chunk or "text" not in chunk:
            raise ValueError(
                f"{dataset_name}: record {index}, chunk {j} must contain title and text."
            )

def load_and_validate_dataset(dataset_name: str, evidence_path: Path) -> List[Dict[str, Any]]:
    """Load and validate one dataset evidence file."""
    records = load_json_list(evidence_path, dataset_name)

    if len(records) != EXPECTED_NUM_RECORDS_PER_DATASET:
        print(
            f"Warning: {dataset_name} has {len(records)} records, "
            f"expected {EXPECTED_NUM_RECORDS_PER_DATASET}."
        )

    for i, rec in enumerate(records):
        validate_evidence_record(rec, dataset_name, i)

    print("=" * 80)
    print(f"{dataset_name}: validation passed.")
    print("Number of records:", len(records))
    print("First question:", records[0]["question"])
    print("First GT answer:", records[0]["answer"])
    print("First number of chunks:", len(records[0]["evidence_chunk"]))
    print("First chunk keys:", list(records[0]["evidence_chunk"][0].keys()))

    return records

evidence_data = {}

for dataset_name, cfg in DATASETS.items():
    evidence_data[dataset_name] = load_and_validate_dataset(
        dataset_name=dataset_name,
        evidence_path=cfg["local_evidence_path"],
    )

hotpotqa: validation passed.
Number of records: 1000
First question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
First GT answer: Bedknobs and Broomsticks
First number of chunks: 4
First chunk keys: ['title', 'text']
2wikimultihopqa: validation passed.
Number of records: 1000
First question: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?
First GT answer: Kamakalawa
First number of chunks: 4
First chunk keys: ['title', 'text']


In [ ]:
# cell 5
# Start GPT-OSS vLLM server on NVIDIA A100 80GB.

def kill_process_tree(pid: int) -> None:
    """Kill a process and all child processes."""
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
        print("Killed process tree:", pid)
    except Exception:
        pass

def find_nvidia_library_dirs() -> List[str]:
    """Find CUDA shared library directories installed by pip packages."""
    dirs = []

    for base in site.getsitepackages():
        pattern = os.path.join(base, "nvidia", "*", "lib")
        for d in glob.glob(pattern):
            if os.path.isdir(d):
                dirs.append(d)

    # Remove duplicates while preserving order.
    unique_dirs = []
    seen = set()

    for d in dirs:
        if d not in seen:
            unique_dirs.append(d)
            seen.add(d)

    return unique_dirs

# Stop old PID from this notebook.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(int(old_pid))

# Stop leftover vLLM serve processes.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(3)

cmd = [
    "vllm", "serve", LLM_MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--tensor-parallel-size", str(TENSOR_PARALLEL_SIZE),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    "--generation-config", "vllm",
    "--trust-remote-code",

    # Let GPT-OSS / MXFP4 use the correct automatic dtype.
    "--dtype", "auto",
]

if KV_CACHE_DTYPE:
    cmd.extend(["--kv-cache-dtype", KV_CACHE_DTYPE])

if MAX_CUDAGRAPH_CAPTURE_SIZE:
    cmd.extend(["--max-cudagraph-capture-size", str(MAX_CUDAGRAPH_CAPTURE_SIZE)])

if not ENABLE_PREFIX_CACHING:
    cmd.append("--no-enable-prefix-caching")
else:
    cmd.append("--enable-prefix-caching")

server_env = os.environ.copy()

# Remove stale Blackwell-specific environment variables from previous runs.
for stale_key in [
    "VLLM_MAIN_CUDA_VERSION",
    "TORCH_CUDA_ARCH_LIST",
    "VLLM_USE_FLASHINFER_MOE_MXFP4_MXFP8",
]:
    server_env.pop(stale_key, None)

# A100 / Ampere attention backend.
if ATTENTION_BACKEND:
    server_env["VLLM_ATTENTION_BACKEND"] = ATTENTION_BACKEND

# Do not set FlashInfer sampler variables on A100 unless explicitly requested.
if USE_FLASHINFER_SAMPLER is not None:
    server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "1" if USE_FLASHINFER_SAMPLER else "0"

if DISABLE_FLASHINFER_COMPLETELY:
    server_env["VLLM_DISABLE_FLASHINFER"] = "1"

# Add pip-installed NVIDIA library paths to help optional CUDA libraries resolve.
if ADD_NVIDIA_PIP_LIBS_TO_LD_LIBRARY_PATH:
    nvidia_lib_dirs = find_nvidia_library_dirs()
    old_ld_path = server_env.get("LD_LIBRARY_PATH", "")
    merged_ld_path = ":".join(nvidia_lib_dirs + ([old_ld_path] if old_ld_path else []))
    server_env["LD_LIBRARY_PATH"] = merged_ld_path

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in [
    "VLLM_ATTENTION_BACKEND",
    "VLLM_USE_FLASHINFER_SAMPLER",
    "VLLM_DISABLE_FLASHINFER",
    "LD_LIBRARY_PATH",
]:
    value = server_env.get(k)

    if k == "LD_LIBRARY_PATH" and value:
        print(f"{k}={value[:500]}{'...' if len(value) > 500 else ''}")
    else:
        print(f"{k}={value}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Command:
vllm serve openai/gpt-oss-120b --host 0.0.0.0 --port 8000 --tensor-parallel-size 1 --max-model-len 49152 --gpu-memory-utilization 0.95 --max-num-seqs 1 --max-num-batched-tokens 1024 --generation-config vllm --trust-remote-code --dtype auto --max-cudagraph-capture-size 1024 --no-enable-prefix-caching

Important environment variables:
VLLM_ATTENTION_BACKEND=TRITON_ATTN
VLLM_USE_FLASHINFER_SAMPLER=None
VLLM_DISABLE_FLASHINFER=None
LD_LIBRARY_PATH=/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvrtc/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_cccl/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cusparse/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cusolver/lib:/usr/local/lib/python3.12/dist-packages/nvidia/nvtx/lib:/usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib:/usr/local/lib/python3.12/dist-packages/nvidia/nccl/lib:/usr/local/lib/pytho...

Started vLLM server.
PID: 2584
Log: /c

In [ ]:
#cell 6
# Wait for vLLM server and create an OpenAI-compatible client.

import requests

def tail_log(path: Path, n: int = 80) -> str:
    """Read the last n log lines."""
    if not path.exists():
        return ""

    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False
SERVER_MODEL_ID = None

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()
    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=5)
        if h.status_code == 200:
            m = requests.get(f"{BASE_URL}/models", timeout=10)
            if m.status_code == 200:
                ready = True
                model_info = m.json()["data"][0]
                SERVER_MODEL_ID = model_info["id"]
                print("vLLM server is ready.")
                print("Model:", SERVER_MODEL_ID)
                print("Max model len:", model_info.get("max_model_len"))
                break
    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")
        recent = tail_log(SERVER_LOG_PATH, n=12)

        if recent.strip():
            print(recent)

        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

# GPT-OSS sampling setup.
LLM_SAMPLING_KWARGS = {
    "temperature": 1.0,
    "top_p": 1.0,
    "presence_penalty": 0.0,
}

# Reasoning effort is passed to the GPT-OSS chat template.
# The QA prompt itself still contains only the question and retrieved contexts.
LLM_EXTRA_BODY = {
    "chat_template_kwargs": {
        "reasoning_effort": REASONING_EFFORT,
    },
    "include_reasoning": True,
}

print("OpenAI-compatible client is ready.")
print("Server model id:", SERVER_MODEL_ID)
print("Reasoning effort:", REASONING_EFFORT)
print("Sampling kwargs:", LLM_SAMPLING_KWARGS)
print("Extra body:", LLM_EXTRA_BODY)

Waiting... 0s
--------------------------------------------------------------------------------
Waiting... 60s
(APIServer pid=2584) WARNING 06-15 18:09:16 [envs.py:2088] Unknown vLLM environment variable detected: VLLM_ATTENTION_BACKEND
(APIServer pid=2584) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(APIServer pid=2584) INFO 06-15 18:09:35 [model.py:611] Resolved architecture: GptOssForCausalLM
(APIServer pid=2584) 
Parse safetensors files: 100%|██████████| 15/15 [00:00<00:00, 18.03it/s]
(APIServer pid=2584) INFO 06-15 18:09:36 [model.py:1745] Using max model len 49152
(APIServer pid=2584) INFO 06-15 18:09:37 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=1024.
(APIServer pid=2584) INFO 06-15 18:09:37 [vllm.py:999] Asynchronous scheduling is enabled.
(APIServer pid=2584) INFO 06-15 18:09:37 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPri

In [ ]:
#cell 7
# Load tokenizer and define prompt/context builders.

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
)

BASE_PROMPT_TEMPLATE = """
Given the question and its associated contexts below, please generate a concise, precise answer in English. The answer must strictly adhere to the following guidelines:

- The answer should be directly relevant to the question.
- Provide the answer in a clear, straightforward format.
- Limit your answer to no more than 6 words, focusing on the essential information requested.
- If the provided contexts do not contain enough information to answer the question, respond with "Information not available".
- Do not include any additional tokens, explanations, or information beyond the direct answer.

QUESTION: {question}
CONTEXT:
{context}

ANSWER:
""".strip()

def count_tokens(text: str) -> int:
    """Count tokens without adding special tokens."""
    return len(tokenizer.encode(text, add_special_tokens=False))

def format_context_from_chunks(evidence_chunks: List[Dict[str, Any]]) -> str:
    """
    Format only the retrieved evidence chunks.
    No answer, supports, or ground-truth fields are included.
    """
    parts = []

    for rank, chunk in enumerate(evidence_chunks, start=1):
        title = str(chunk.get("title", "")).strip()
        text = str(chunk.get("text", "")).strip()

        parts.append(
            f"[Chunk {rank}]\n"
            f"Title: {title}\n"
            f"Text: {text}"
        )

    return "\n\n".join(parts).strip()

def build_prompt_without_truncation(question: str, context: str) -> str:
    """Build the final user prompt."""
    return BASE_PROMPT_TEMPLATE.format(
        question=str(question).strip(),
        context=str(context).strip(),
    )

def build_prompt(question: str, evidence_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Build a prompt using only question and evidence_chunk.
    The context is truncated if the full prompt exceeds MAX_INPUT_TOKENS.
    """
    context = format_context_from_chunks(evidence_chunks)
    prompt = build_prompt_without_truncation(question, context)

    original_tokens = count_tokens(prompt)

    if original_tokens <= MAX_INPUT_TOKENS:
        return {
            "prompt": prompt,
            "input_tokens": original_tokens,
            "was_truncated": False,
        }

    # Compute available token budget for context after keeping the prompt shell.
    empty_prompt = build_prompt_without_truncation(question, "")
    overhead_tokens = count_tokens(empty_prompt)

    context_budget = max(0, MAX_INPUT_TOKENS - overhead_tokens - 16)

    context_ids = tokenizer.encode(context, add_special_tokens=False)
    truncated_context_ids = context_ids[:context_budget]
    truncated_context = tokenizer.decode(truncated_context_ids, skip_special_tokens=True)

    truncated_prompt = build_prompt_without_truncation(question, truncated_context)
    truncated_tokens = count_tokens(truncated_prompt)

    return {
        "prompt": truncated_prompt,
        "input_tokens": truncated_tokens,
        "was_truncated": True,
        "original_input_tokens": original_tokens,
    }

# Sanity check: make sure the prompt does not contain forbidden fields by construction.
sample_dataset = "hotpotqa"
sample_record = evidence_data[sample_dataset][0]
sample_prompt_info = build_prompt(
    question=sample_record["question"],
    evidence_chunks=sample_record["evidence_chunk"],
)

print("Sample dataset:", sample_dataset)
print("Sample input tokens:", sample_prompt_info["input_tokens"])
print("Sample was truncated:", sample_prompt_info["was_truncated"])
print("\nSample prompt preview:")
print(sample_prompt_info["prompt"][:2000])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Sample dataset: hotpotqa
Sample input tokens: 1798
Sample was truncated: False

Sample prompt preview:
Given the question and its associated contexts below, please generate a concise, precise answer in English. The answer must strictly adhere to the following guidelines:

- The answer should be directly relevant to the question.
- Provide the answer in a clear, straightforward format.
- Limit your answer to no more than 6 words, focusing on the essential information requested.
- If the provided contexts do not contain enough information to answer the question, respond with "Information not available".
- Do not include any additional tokens, explanations, or information beyond the direct answer.

QUESTION: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
CONTEXT:
[Chunk 1]
Title: The Muppet Christmas Carol
Text: The Muppet Christmas Carol is a 1992 American-British musical fantasy comedy-drama film and an adaptation of Charles Dickens's 1843 n

In [ ]:
#cell 8
# Define LLM call, response cleanup, and JSON saving helpers.

def clean_model_response(text: str) -> str:
    """Clean possible reasoning, harmony, or formatting artifacts from the model output."""
    if text is None:
        return ""

    text = str(text)

    # Remove possible XML-style thinking blocks.
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = text.replace("<think>", "").replace("</think>", "")

    # Remove possible harmony-like special tokens if they appear in text.
    text = re.sub(r"<\|[^>]+?\|>", "", text)

    text = text.strip()

    # Remove common labels if the model adds them.
    text = re.sub(r"^\s*ANSWER\s*:\s*", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"^\s*FINAL\s*:\s*", "", text, flags=re.IGNORECASE).strip()

    # Remove markdown code fences if any.
    text = text.replace("```json", "").replace("```text", "").replace("```", "").strip()

    # Keep only the first non-empty line.
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        text = lines[0].strip()

    # Remove surrounding quotes.
    text = text.strip().strip('"').strip("'").strip()

    return text

def extract_chat_message_content(completion: Any) -> str:
    """Extract final message content from an OpenAI-compatible chat completion."""
    message = completion.choices[0].message

    content = getattr(message, "content", None)

    if content is None and isinstance(message, dict):
        content = message.get("content")

    # We intentionally do not save reasoning_content.
    # Only the final assistant content is used as response.
    if content is None:
        content = ""

    return str(content)

def call_llm_answer(prompt: str, max_retries: int = 3) -> str:
    """Call the local vLLM OpenAI-compatible server and return a cleaned final answer."""
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            completion = client.chat.completions.create(
                model=SERVER_MODEL_ID or LLM_MODEL_NAME,
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                max_tokens=ANSWER_MAX_TOKENS,
                **LLM_SAMPLING_KWARGS,
                extra_body=LLM_EXTRA_BODY,
            )

            raw_text = extract_chat_message_content(completion)
            cleaned = clean_model_response(raw_text)

            if cleaned:
                return cleaned

            print(f"Warning: empty final answer on attempt {attempt}/{max_retries}. Retrying...")
            time.sleep(2 * attempt)

        except Exception as e:
            last_error = e
            print(f"LLM call failed on attempt {attempt}/{max_retries}: {repr(e)}")
            time.sleep(2 * attempt)

    if last_error is not None:
        raise RuntimeError(f"LLM call failed after {max_retries} attempts: {repr(last_error)}")

    return ""

def atomic_write_json(data: Any, path: Path) -> None:
    """Atomically save JSON to the target path."""
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = path.with_name(path.name + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    os.replace(tmp_path, path)

def load_existing_answers(answer_path: Path) -> List[Dict[str, Any]]:
    """Load existing answers for resume mode."""
    if not answer_path.exists():
        return []

    with open(answer_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"Existing answer file must contain a list: {answer_path}")

    return data

def validate_answer_record(record: Dict[str, Any], dataset_name: str, index: int) -> None:
    """Validate one answer output record."""
    required_keys = {"type", "question", "gt", "response"}

    if set(record.keys()) != required_keys:
        raise ValueError(
            f"{dataset_name}: answer record {index} must have exactly {required_keys}, "
            f"but found {set(record.keys())}"
        )

print("Helper functions are ready.")

Helper functions are ready.


In [ ]:
#cell 9
# Test one LLM call before running the full datasets.

test_dataset = "hotpotqa"
test_record = evidence_data[test_dataset][0]

test_prompt_info = build_prompt(
    question=test_record["question"],
    evidence_chunks=test_record["evidence_chunk"],
)

print("Test dataset:", test_dataset)
print("Test question:", test_record["question"])
print("Test GT answer:", test_record["answer"])
print("Note: GT is printed only for human checking and is not sent to the LLM.")
print("Test input tokens:", test_prompt_info["input_tokens"])
print("Test was truncated:", test_prompt_info["was_truncated"])

test_response = call_llm_answer(test_prompt_info["prompt"])

print("Test LLM response:", test_response)

Test dataset: hotpotqa
Test question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
Test GT answer: Bedknobs and Broomsticks
Note: GT is printed only for human checking and is not sent to the LLM.
Test input tokens: 1798
Test was truncated: False
Test LLM response: Bedknobs and Broomsticks is older.


In [ ]:
#cell 10
# Process one dataset and save answers.

def get_processing_slice(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Return the selected processing slice."""
    end = ANSWER_END_INDEX if ANSWER_END_INDEX is not None else len(records)
    return records[ANSWER_START_INDEX:end]

def process_dataset(dataset_name: str, records: List[Dict[str, Any]], answer_path: Path) -> Dict[str, Any]:
    """
    Process one dataset independently.

    Important:
    - Only question and evidence_chunk are given to the LLM.
    - answer, supports, type, and any other fields are not included in the prompt.
    - The final saved JSON contains exactly: type, question, gt, response.
    """
    print("=" * 80)
    print(f"Starting dataset: {dataset_name}")
    print("Answer path:", answer_path)

    selected_records = get_processing_slice(records)

    if not selected_records:
        raise ValueError(f"{dataset_name}: selected record slice is empty.")

    answer_records = []

    if RESUME_IF_EXISTS and answer_path.exists():
        existing = load_existing_answers(answer_path)

        # Resume only if the existing file matches the selected prefix.
        can_resume = True

        if len(existing) > len(selected_records):
            can_resume = False
        else:
            for i, old in enumerate(existing):
                if old.get("question") != selected_records[i].get("question"):
                    can_resume = False
                    break

        if can_resume:
            answer_records = existing
            print(f"{dataset_name}: resuming from {len(answer_records)} existing answers.")
        else:
            print(f"{dataset_name}: existing answer file does not match current data. Starting over.")

    start_time = time.time()

    input_token_counts = []
    truncation_count = 0

    start_i = len(answer_records)

    for local_i in tqdm(
        range(start_i, len(selected_records)),
        desc=f"Answering {dataset_name}"
    ):
        rec = selected_records[local_i]

        prompt_info = build_prompt(
            question=rec["question"],
            evidence_chunks=rec["evidence_chunk"],
        )

        input_token_counts.append(prompt_info["input_tokens"])

        if prompt_info.get("was_truncated", False):
            truncation_count += 1

        response = call_llm_answer(prompt_info["prompt"])

        output_record = {
            "type": rec["type"],
            "question": rec["question"],
            "gt": rec["answer"],
            "response": response,
        }

        validate_answer_record(output_record, dataset_name, local_i)
        answer_records.append(output_record)

        if len(answer_records) % SAVE_EVERY_N == 0:
            atomic_write_json(answer_records, answer_path)

        if CLEAR_CACHE_EVERY_N and len(answer_records) % CLEAR_CACHE_EVERY_N == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    atomic_write_json(answer_records, answer_path)

    elapsed = time.time() - start_time

    # Validate saved file.
    saved = load_existing_answers(answer_path)

    if len(saved) != len(selected_records):
        raise ValueError(
            f"{dataset_name}: saved {len(saved)} answers, "
            f"but expected {len(selected_records)}."
        )

    for i, out_rec in enumerate(saved):
        validate_answer_record(out_rec, dataset_name, i)

    summary = {
        "dataset": dataset_name,
        "num_input_records": len(records),
        "num_processed_records": len(saved),
        "answer_path": str(answer_path),
        "elapsed_seconds": elapsed,
        "num_truncated_prompts": truncation_count,
        "max_input_tokens_seen": max(input_token_counts) if input_token_counts else None,
        "mean_input_tokens_seen": float(sum(input_token_counts) / len(input_token_counts)) if input_token_counts else None,
    }

    print("=" * 80)
    print(f"{dataset_name}: done.")
    print("Saved to:", answer_path)
    print("Processed records:", len(saved))
    print("Elapsed seconds:", elapsed)
    print("Truncated prompts:", truncation_count)
    print("First output record:", saved[0])

    return summary

print("Dataset processing function is ready.")

Dataset processing function is ready.


In [ ]:
# cell 11
# Run HotpotQA separately and save its BM25-based answer JSON file.

hotpotqa_summary = process_dataset(
    dataset_name="hotpotqa",
    records=evidence_data["hotpotqa"],
    answer_path=DATASETS["hotpotqa"]["answer_path"],
)

hotpotqa_summary_df = pd.DataFrame([hotpotqa_summary])
display(hotpotqa_summary_df)

print("HotpotQA BM25 run is complete.")
print("Saved to:", DATASETS["hotpotqa"]["answer_path"])

Starting dataset: hotpotqa
Answer path: /content/drive/MyDrive/final_project/BM25/answer/hotpotqa_bm25_gpt_oss_120b_answers.json


Answering hotpotqa:   0%|          | 0/1000 [00:00<?, ?it/s]

hotpotqa: done.
Saved to: /content/drive/MyDrive/final_project/BM25/answer/hotpotqa_bm25_gpt_oss_120b_answers.json
Processed records: 1000
Elapsed seconds: 1354.908767938614
Truncated prompts: 0
First output record: {'type': 'comparison', 'question': 'Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?', 'gt': 'Bedknobs and Broomsticks', 'response': 'Bedknobs and Broomsticks is older'}


,dataset,num_input_records,num_processed_records,answer_path,elapsed_seconds,num_truncated_prompts,max_input_tokens_seen,mean_input_tokens_seen
0,hotpotqa,1000,1000,/content/drive/MyDrive/final_project/BM25/answ...,1354.908768,0,2150,1584.962


HotpotQA BM25 run is complete.
Saved to: /content/drive/MyDrive/final_project/BM25/answer/hotpotqa_bm25_gpt_oss_120b_answers.json


In [12]:
# cell 12
# Run 2WikiMultiHopQA separately and save its BM25-based answer JSON file.

twowiki_summary = process_dataset(
    dataset_name="2wikimultihopqa",
    records=evidence_data["2wikimultihopqa"],
    answer_path=DATASETS["2wikimultihopqa"]["answer_path"],
)

twowiki_summary_df = pd.DataFrame([twowiki_summary])
display(twowiki_summary_df)

print("2WikiMultiHopQA BM25 run is complete.")
print("Saved to:", DATASETS["2wikimultihopqa"]["answer_path"])

Starting dataset: 2wikimultihopqa
Answer path: /content/drive/MyDrive/final_project/BM25/answer/2wikimultihopqa_bm25_gpt_oss_120b_answers.json


Answering 2wikimultihopqa:   0%|          | 0/1000 [00:00<?, ?it/s]

2wikimultihopqa: done.
Saved to: /content/drive/MyDrive/final_project/BM25/answer/2wikimultihopqa_bm25_gpt_oss_120b_answers.json
Processed records: 1000
Elapsed seconds: 1288.6067423820496
Truncated prompts: 0
First output record: {'type': 'bridge_comparison', 'question': 'Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?', 'gt': 'Kamakalawa', 'response': 'Information not available'}


,dataset,num_input_records,num_processed_records,answer_path,elapsed_seconds,num_truncated_prompts,max_input_tokens_seen,mean_input_tokens_seen
0,2wikimultihopqa,1000,1000,/content/drive/MyDrive/final_project/BM25/answ...,1288.606742,0,2168,1430.016


2WikiMultiHopQA BM25 run is complete.
Saved to: /content/drive/MyDrive/final_project/BM25/answer/2wikimultihopqa_bm25_gpt_oss_120b_answers.json


In [13]:
# cell 13
# Verify final BM25 output files.

def verify_final_answer_file(dataset_name: str, answer_path: Path) -> None:
    """Verify final answer JSON format."""
    records = load_existing_answers(answer_path)

    if not records:
        raise ValueError(f"{dataset_name}: answer file is empty: {answer_path}")

    for i, rec in enumerate(records):
        validate_answer_record(rec, dataset_name, i)

    print("=" * 80)
    print(f"{dataset_name}: final answer file verified.")
    print("Path:", answer_path)
    print("Number of records:", len(records))
    print("First record keys:", list(records[0].keys()))
    print("First question:", records[0]["question"])
    print("First GT:", records[0]["gt"])
    print("First response:", records[0]["response"])

for dataset_name, cfg in DATASETS.items():
    verify_final_answer_file(dataset_name, cfg["answer_path"])

print("\nBM25 answer folder contents:")
for item in sorted(DRIVE_ANSWER_DIR.iterdir()):
    print(" -", item.name)

hotpotqa: final answer file verified.
Path: /content/drive/MyDrive/final_project/BM25/answer/hotpotqa_bm25_gpt_oss_120b_answers.json
Number of records: 1000
First record keys: ['type', 'question', 'gt', 'response']
First question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
First GT: Bedknobs and Broomsticks
First response: Bedknobs and Broomsticks is older
2wikimultihopqa: final answer file verified.
Path: /content/drive/MyDrive/final_project/BM25/answer/2wikimultihopqa_bm25_gpt_oss_120b_answers.json
Number of records: 1000
First record keys: ['type', 'question', 'gt', 'response']
First question: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?
First GT: Kamakalawa
First response: Information not available

BM25 answer folder contents:
 - 2wikimultihopqa_bm25_gpt_oss_120b_answers.json
 - 2wikimultihopqa_qwen3.5_bm25_answers.json
 - hotpotqa_bm25_gpt_oss_120b_answers.json
 - hotpotqa_qwen3.5_bm25_answers.j